In [1]:
! pip install bs4 requests

In [2]:
from pprint import pprint

# Scrape books web-scraping training page


In [3]:
import requests

url = 'https://books.toscrape.com/'

# Select a random agent for the request
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9",
}

# Perform the request
response = requests.get(url, headers=headers)
print(response.status_code)
print(response.text[:500])

200
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


In [4]:
import bs4
# formatter to be passed to soup.prettify() to improve readibility
formatter = bs4.formatter.HTMLFormatter(indent=3)

In [5]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")  # Parse the HTML content of the page

In [6]:
result = soup.find_all("li.article")
result

[]

In [7]:
rows = soup.find_all(
    "article", 
    class_="product_pod",
)
print(len(rows),
    #"\n", rows[0].prettify(formatter=formatter)
)

20


In [8]:
rows = soup.select("article.product_pod")  # CSS selector for article tags with class product_pod
print(len(rows),
    #"\n", rows[0].prettify(formatter=formatter)
)

20


In [9]:
rows = soup.select(".product_pod")  # CSS selector for all tags with class product_pod
print(len(rows),
    #"\n", rows[0].prettify(formatter=formatter)
)

20


In [10]:
_ = rows[0].a  # first tag <a> in rows[0] --> not the one we need
print(_.prettify(formatter=formatter))

<a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
</a>



In [11]:
_ = rows[0].h3.a  # a tag in the h3 tag section
print(_.prettify(formatter=formatter))

<a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
</a>



In [12]:
rows[0].find(".title")  # does not work

In [13]:
rows[0].h3.a.title  # does not work

In [14]:
rows[0].h3.a["title"]

'A Light in the Attic'

In [15]:
rows[0].find("h3").a["title"]


'A Light in the Attic'

In [16]:
soup.find_all("h3")[:3] # .a["title"]

[<h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>,
 <h3><a href="catalogue/tipping-the-velvet_999/index.html" title="Tipping the Velvet">Tipping the Velvet</a></h3>,
 <h3><a href="catalogue/soumission_998/index.html" title="Soumission">Soumission</a></h3>]

In [17]:
titles = [row.find("h3").a["title"] for row in rows]
titles[:3]

['A Light in the Attic', 'Tipping the Velvet', 'Soumission']

In [18]:
#print(rows[0].prettify(formatter=formatter))

In [19]:
p = rows[0].select_one(".price_color").text
p

'Â£51.77'

In [20]:
prices = [row.find("p", class_="price_color").text for row in rows]
prices[:3]


['Â£51.77', 'Â£53.74', 'Â£50.10']

In [21]:
import re
prices = [float(re.search(r"(\d+.\d{2})", p).group()) for p in prices]
prices[:3]

[51.77, 53.74, 50.1]

In [22]:
import pandas as pd
df = pd.DataFrame({
    "Title": titles,
    "Price": prices,
})
df.head(), df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Title   20 non-null     str    
 1   Price   20 non-null     float64
dtypes: float64(1), str(1)
memory usage: 1.1 KB


(                                   Title  Price
 0                   A Light in the Attic  51.77
 1                     Tipping the Velvet  53.74
 2                             Soumission  50.10
 3                          Sharp Objects  47.82
 4  Sapiens: A Brief History of Humankind  54.23,
 None)

In [23]:
df.to_csv("books_page1.csv", index=False)